In [ ]:
import importlib
from pathlib import Path
from torch.utils.data import DataLoader
import h5py
import time
import numpy as np
from functions_library import *

In [ ]:
test_dataset = dataset_classes.H5EgammaDataset(
    h5_files_egam,
    mix_files = h5_files_data,
    mixture_ratio = 0.8,
    exclude_features = ["truth"],
    exclude_fields = ["truthOrigin", "truthType"],
    fallback_target = ["passid_Medium"],
    y_source = "egammas",
    y_field = "truthOrigin",
)
sampler = data_loading.ShuffledContiguousBatchSampler(len(test_dataset), batch_size=1024, dataset=test_dataset)
test_loader = DataLoader(test_dataset, batch_sampler=sampler, num_workers=8, prefetch_factor=2, persistent_workers=True)
for batch_idx, (_) in enumerate(test_loader):
    if batch_idx == 5:
        start = time.perf_counter()
    if batch_idx == 25:
        end = time.perf_counter()
        break
print(f"Time taken to load 10 batches: {end - start:.4f} seconds")

In [ ]:
dataset_mean = dataset_classes.H5EgammaDataset(
    h5_files_egam[1],
    exclude_features=["truth"],
    exclude_fields=["truthOrigin", "truthType", "pt"],
)
means, stds = misc_functions.compute_mean_std(dataset_mean, batch_size=1024, num_workers=4, sample_size=10000)
dataset_mean.close()

dataset_origin = dataset_classes.ZPairDataset(
    h5_files_egam[1],
    exclude_features=["truth"],
    exclude_fields=["truthOrigin", "truthType", "pt"],
    y_source="truth",
    y_field=["pt"]
)
origin_loader = DataLoader(dataset_origin, batch_size=1024, shuffle=False, num_workers=4, prefetch_factor=2, persistent_workers=True)

time_start_o = time.perf_counter()
_ = model_training.validate_model(models["pt"], training_classes.GaussianNLLLoss(), origin_loader, device=device, means=means, stds=stds, log_target=True)
time_o = time.perf_counter() - time_start_o

time_start_p = time.perf_counter()
_ = model_training.predict(models["pt"], origin_loader, device=device, log_target=True, means=means, stds=stds)
time_p = time.perf_counter() - time_start_p
dataset_origin.close()

print(f"Time taken to validate model: {time_o:.2f} seconds")
print(f"Time taken to predict model: {time_p:.2f} seconds")